In [2]:
# pip install biopython

In [3]:
import os
import glob
import pandas as pd
from Bio import PDB
from Bio.SeqUtils import seq1

In [4]:
# 1. route dir
raw_pdb_dir = os.path.join("MCGLPPI_RawData", "pdbs", "m2_pdbbind_dimer_strict")
fasta_out_dir = os.path.join("ProAffinity-GNN", "data", "FASTA_", "mixed")

# cache file
clean_pdb_dir = os.path.join("ProAffinity-GNN", "data", "temp_pdb")


csv_path = os.path.join("MCGLPPI_RawData", "PDBBINDdimer_strict_index.csv")

os.makedirs(fasta_out_dir, exist_ok=True)
os.makedirs(clean_pdb_dir, exist_ok=True)

In [5]:
# 2. Read
df = pd.read_csv(csv_path)
pdb_list = df['pdb_code'].dropna().unique().tolist()

parser = PDB.PDBParser(QUIET=True)
success_count = 0

print(f"Typecasting {len(pdb_list)} files...")

for pdb_id in pdb_list:
    # Find target pdb
    target_folder = os.path.join(raw_pdb_dir, pdb_id)
    if not os.path.exists(target_folder):
        print(f"No such file: {pdb_id}")
        continue
        
    # Find pdb file, ignoring -cg
    pdb_files = [f for f in glob.glob(os.path.join(target_folder, "*.pdb")) if "-cg" not in f.lower()]
    
    if not pdb_files:
        print(f"No all atom pdb file for {pdb_id}")
        continue
        
    pdb_file_path = pdb_files[0] 
    
    # ---------------------------------------------------
    # Task A: FASTA (ProAffinity)
    # ---------------------------------------------------
    structure = parser.get_structure(pdb_id, pdb_file_path)
    chain_idx = 1
    
    for model in structure:
        for chain in model:
            # Extract sequence
            seq = ""
            for residue in chain:
                if PDB.is_aa(residue, standard=True):
                    seq += seq1(residue.resname)
            
            if len(seq) > 0:
                # Naming rule: 1A3B_1.fasta, 1A3B_2.fasta (Block letter PDB_no.)
                fasta_name = f"{pdb_id.upper()}_{chain_idx}.fasta"
                fasta_path = os.path.join(fasta_out_dir, fasta_name)
                
                with open(fasta_path, "w") as f:
                    f.write(f">{pdb_id.upper()}:{chain.id}\n")
                    f.write(f"{seq}\n")
                
                chain_idx += 1
                
    # ---------------------------------------------------
    # Task B: Duplicate pdb and rename
    # ---------------------------------------------------
    # Standardize as pdb_id.pdb, and save in temp_pdb
    import shutil
    clean_pdb_path = os.path.join(clean_pdb_dir, f"{pdb_id}.pdb")
    shutil.copy(pdb_file_path, clean_pdb_path)
    
    success_count += 1

print(f"\n Number of successful typecasting to fasta protein:{success_count}")
print(f"FASTA sequence saved in : {fasta_out_dir}")
print(f"PDB saved in : {clean_pdb_dir}")

Typecasting 1270 files...

 Number of successful typecasting to fasta protein:1270
FASTA sequence saved in : ProAffinity-GNN\data\FASTA_\mixed
PDB saved in : ProAffinity-GNN\data\temp_pdb
